In [1]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

try:
    from google.protobuf import message_factory as _message_factory

    if hasattr(_message_factory, "MessageFactory") and not hasattr(
        _message_factory.MessageFactory, "GetPrototype"
    ):

        def _GetPrototype(self, descriptor):
            return self.GetMessageClass(descriptor)

        _message_factory.MessageFactory.GetPrototype = _GetPrototype
except Exception:
    pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

import tensorflow as tf

try:
    tf.config.experimental.enable_tensor_float_32_execution(False)
except Exception:
    pass

from tf_keras.models import Sequential
from tf_keras.layers import Dense, BatchNormalization
from tf_keras import optimizers, regularizers

np.random.seed(1)
tf.random.set_seed(1)

TRAIN_PATH = "/kaggle/input/new-york-city-taxi-fare-prediction/train.csv"
TEST_PATH = "/kaggle/input/new-york-city-taxi-fare-prediction/test.csv"
SUBMISSION_NAME = "submissiontry_water.csv"

BATCH_SIZE = 256
EPOCHS = 30
LEARNING_RATE = 0.001

DATASET_SIZE = 250000

USE_LOG_TARGET = False

if not os.path.exists(TRAIN_PATH):
    alt = "/kaggle/input/train.csv"
    if os.path.exists(alt):
        TRAIN_PATH = alt
    else:
        TRAIN_PATH = "../input/train.csv"

if not os.path.exists(TEST_PATH):
    alt = "/kaggle/input/test.csv"
    if os.path.exists(alt):
        TEST_PATH = alt
    else:
        TEST_PATH = "../input/test.csv"

print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)

if not SUBMISSION_NAME.lower().endswith(".csv"):
    SUBMISSION_NAME = SUBMISSION_NAME + ".csv"
SUBMISSION_PATH = os.path.join("/kaggle/working", SUBMISSION_NAME)
print("SUBMISSION_PATH:", SUBMISSION_PATH)



2026-01-17 22:41:05.370835: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768689665.384244      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768689665.388348      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TRAIN_PATH: /kaggle/input/new-york-city-taxi-fare-prediction/train.csv
TEST_PATH: /kaggle/input/new-york-city-taxi-fare-prediction/test.csv
SUBMISSION_PATH: /kaggle/working/submissiontry_water.csv


In [2]:
datatypes = {
    "key": "str",
    "fare_amount": "float32",
    "pickup_datetime": "str",
    "pickup_longitude": "float32",
    "pickup_latitude": "float32",
    "dropoff_longitude": "float32",
    "dropoff_latitude": "float32",
    "passenger_count": "uint8",
}

trainKaggle = pd.read_csv(
    TRAIN_PATH,
    nrows=DATASET_SIZE,
    dtype=datatypes,
    usecols=[
        "key",
        "fare_amount",
        "pickup_datetime",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "passenger_count",
    ],
)

testKaggle = pd.read_csv(
    TEST_PATH,
    dtype={
        "key": "str",
        "pickup_datetime": "str",
        "pickup_longitude": "float32",
        "pickup_latitude": "float32",
        "dropoff_longitude": "float32",
        "dropoff_latitude": "float32",
        "passenger_count": "uint8",
    },
)



In [3]:
train_df, validation_df = train_test_split(trainKaggle, test_size=0.10, random_state=1)



In [4]:
print("testKaggle Size %d" % len(testKaggle))
print("train_df Size %d" % len(train_df))
print("validation_df Size %d" % len(validation_df))



testKaggle Size 9914
train_df Size 225000
validation_df Size 25000


In [5]:
train_df.describe()



,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,225000.000000,225000.000000,225000.000000,224998.000000,224998.000000,225000.000000
mean,11.358297,-72.521156,39.925129,-72.522812,39.926804,1.682876
std,9.888828,11.630339,9.683873,10.678661,6.697321,1.307365
min,-44.900002,-740.000000,-3116.285400,-1251.195923,-1189.615479,0.000000
25%,6.000000,-73.992035,40.735012,-73.991310,40.734116,1.000000
50%,8.500000,-73.981766,40.752743,-73.980057,40.753220,1.000000
75%,12.500000,-73.967056,40.767143,-73.963509,40.768093,2.000000
max,495.000000,2140.601074,1703.092773,40.825310,404.616669,6.000000


In [6]:
validation_df.describe()




,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000
mean,11.353762,-72.553177,39.947861,-72.558601,39.947811,1.687680
std,10.375725,10.267632,6.013676,10.245173,6.013325,1.307411
min,-2.900000,-75.336082,-74.005409,-75.362427,-74.008247,0.000000
25%,6.000000,-73.992081,40.734762,-73.991455,40.733504,1.000000
50%,8.500000,-73.981777,40.753057,-73.980347,40.753170,1.000000
75%,12.500000,-73.967285,40.767102,-73.963928,40.767883,2.000000
max,500.000000,40.788326,41.366138,40.851028,41.366138,6.000000


In [7]:
def remove_datapoints_from_water(df):
    """
    Score-relevant fix (minimal): when the NYC water mask image is not available offline,
    do NOT apply the previous heuristic filter (it can remove many valid airport/coastal trips
    and worsen RMSE). Instead, fall back to a no-op so we only use deterministic NYC bounding-box
    cleaning already applied elsewhere.
    """

    def lonlat_to_xy(longitude, latitude, dx, dy, BB):
        x = (dx * (longitude - BB[0]) / (BB[1] - BB[0])).astype("int")
        y = (dy - dy * (latitude - BB[2]) / (BB[3] - BB[2])).astype("int")
        return x, y

    BB = (-74.5, -72.8, 40.5, 41.8)

    local_mask_paths = [
        "/kaggle/input/nyc_mask-74.5_-72.8_40.5_41.8.png",
        "/kaggle/working/nyc_mask-74.5_-72.8_40.5_41.8.png",
        "nyc_mask-74.5_-72.8_40.5_41.8.png",
    ]

    mask_path = None
    for p in local_mask_paths:
        if os.path.exists(p):
            mask_path = p
            break

    if mask_path is None:
        # No-op fallback to avoid over-filtering and score degradation.
        return df

    nyc_mask = plt.imread(mask_path)
    if nyc_mask.ndim == 3:
        nyc_mask = nyc_mask[:, :, 0]
    nyc_mask = nyc_mask > 0.9

    pickup_x, pickup_y = lonlat_to_xy(
        df.pickup_longitude,
        df.pickup_latitude,
        nyc_mask.shape[1],
        nyc_mask.shape[0],
        BB,
    )
    dropoff_x, dropoff_y = lonlat_to_xy(
        df.dropoff_longitude,
        df.dropoff_latitude,
        nyc_mask.shape[1],
        nyc_mask.shape[0],
        BB,
    )

    pickup_x = np.clip(pickup_x, 0, nyc_mask.shape[1] - 1)
    dropoff_x = np.clip(dropoff_x, 0, nyc_mask.shape[1] - 1)
    pickup_y = np.clip(pickup_y, 0, nyc_mask.shape[0] - 1)
    dropoff_y = np.clip(dropoff_y, 0, nyc_mask.shape[0] - 1)

    in_water = nyc_mask[pickup_y, pickup_x] | nyc_mask[dropoff_y, dropoff_x]
    return df[~in_water]


def clean(df):
    print(" Old size: %d" % len(df))
    df = df.dropna(how="any", axis="rows")
    print(" New size after dropna: %d" % len(df))

    df = df[
        ~(
            (df["dropoff_longitude"] == df["pickup_longitude"])
            & (df["dropoff_latitude"] == df["pickup_latitude"])
        )
    ]
    print(" New size after removing same pickup/dropoff point: %d" % len(df))

    df = df[
        (df["dropoff_longitude"] != 0)
        & (df["pickup_longitude"] != 0)
        & (df["dropoff_latitude"] != 0)
        & (df["pickup_latitude"] != 0)
    ]
    print(" New size after removing 0 long lat: %d" % len(df))

    MinMax = (-74.5, -72.8, 40.5, 41.8)
    df = df[
        (MinMax[0] <= df["pickup_longitude"]) & (df["pickup_longitude"] <= MinMax[1])
    ]
    df = df[
        (MinMax[0] <= df["dropoff_longitude"]) & (df["dropoff_longitude"] <= MinMax[1])
    ]
    df = df[(MinMax[2] <= df["pickup_latitude"]) & (df["pickup_latitude"] <= MinMax[3])]
    df = df[
        (MinMax[2] <= df["dropoff_latitude"]) & (df["dropoff_latitude"] <= MinMax[3])
    ]
    print(" New size after only NYC: %d" % len(df))

    if "fare_amount" in df.columns:
        df = df[(0 < df["fare_amount"]) & (df["fare_amount"] <= 50)]
        print(" New size after removing outliers: %d" % len(df))

    df = df[(df["passenger_count"] > 0) & (df["passenger_count"] <= 6)]
    print(" New size after removing 6=>passenger_count > 0 : %d" % len(df))

    print(" Skipping landmark point-noise removal (was overly aggressive).")

    print("Old size: %d" % len(df))
    df = remove_datapoints_from_water(df)
    print("New size: %d" % len(df))

    print(" New size: %d" % len(df))
    return df


def manhattan(pickup_lat, pickup_long, dropoff_lat, dropoff_long):
    return np.abs(dropoff_lat - pickup_lat) + np.abs(dropoff_long - pickup_long)


def add_time_features(df):
    dt = pd.to_datetime(df["pickup_datetime"], errors="coerce", utc=True)

    if "fare_amount" in df.columns:
        valid = dt.notna()
        if valid.sum() != len(df):
            df = df.loc[valid].copy()
            dt = dt.loc[valid]
    else:
        dt = dt.fillna(pd.Timestamp("2010-01-01", tz="UTC"))

    dt = dt.dt.tz_convert(None)

    df["year"] = dt.dt.year.astype("float32")
    df["month"] = dt.dt.month.astype("float32")
    df["day"] = dt.dt.day.astype("float32")
    df["hour"] = dt.dt.hour.astype("float32")
    df["weekday"] = dt.dt.weekday.astype("float32")

    df["pickup_datetime"] = dt.astype(str)

    weekday = df["weekday"]
    hour = df["hour"]

    df["night"] = ((weekday < 5) & ((hour >= 20) | (hour <= 6))).astype("uint8")
    df["late_night"] = (hour <= 3).astype("uint8")
    df["rush_hour"] = ((weekday < 5) & (hour >= 16) & (hour <= 20)).astype("uint8")

    return df


def add_coordinate_features(df):
    lat1 = df["pickup_latitude"]
    lat2 = df["dropoff_latitude"]
    lon1 = df["pickup_longitude"]
    lon2 = df["dropoff_longitude"]
    df["latdiff"] = (lat1 - lat2).astype("float32")
    df["londiff"] = (lon1 - lon2).astype("float32")
    return df


def add_distances_features(df):
    lat1 = df["pickup_latitude"]
    lat2 = df["dropoff_latitude"]
    lon1 = df["pickup_longitude"]
    lon2 = df["dropoff_longitude"]
    df["manhattan"] = manhattan(lat1, lon1, lat2, lon2).astype("float32")
    df["distance"] = np.sqrt(
        np.abs(df["pickup_longitude"] - df["dropoff_longitude"]) ** 2
        + np.abs(df["pickup_latitude"] - df["dropoff_latitude"]) ** 2
    ).astype("float32")
    return df


def _haversine_km(lon1, lat1, lon2, lat2):
    # Vectorized haversine in kilometers (stable, no extra deps).
    lon1 = np.radians(lon1.astype("float64"))
    lat1 = np.radians(lat1.astype("float64"))
    lon2 = np.radians(lon2.astype("float64"))
    lat2 = np.radians(lat2.astype("float64"))
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.minimum(1.0, np.sqrt(a)))
    return (6371.0 * c).astype("float32")


def add_airport_features(df):
    """
    Score-relevant improvement (minimal): add airport distance signals which strongly correlate
    with fare in this competition, without changing the model/training loop.
    """
    # Airport coordinates (approx)
    JFK = (-73.7781, 40.6413)
    LGA = (-73.8740, 40.7769)
    EWR = (-74.1745, 40.6895)

    plon = df["pickup_longitude"]
    plat = df["pickup_latitude"]
    dlon = df["dropoff_longitude"]
    dlat = df["dropoff_latitude"]

    # Trip distance in km (better scale than raw degrees)
    df["haversine_km"] = _haversine_km(plon, plat, dlon, dlat)

    # Min distance to major airports from pickup/dropoff
    p_jfk = _haversine_km(plon, plat, JFK[0], JFK[1])
    p_lga = _haversine_km(plon, plat, LGA[0], LGA[1])
    p_ewr = _haversine_km(plon, plat, EWR[0], EWR[1])
    d_jfk = _haversine_km(dlon, dlat, JFK[0], JFK[1])
    d_lga = _haversine_km(dlon, dlat, LGA[0], LGA[1])
    d_ewr = _haversine_km(dlon, dlat, EWR[0], EWR[1])

    df["pickup_airport_km"] = np.minimum(np.minimum(p_jfk, p_lga), p_ewr).astype(
        "float32"
    )
    df["dropoff_airport_km"] = np.minimum(np.minimum(d_jfk, d_lga), d_ewr).astype(
        "float32"
    )

    return df


def output_submission(raw_test, prediction, id_column, prediction_column, file_name):
    df = pd.DataFrame(
        {
            id_column: raw_test[id_column].astype(str),
            prediction_column: prediction.reshape(-1),
        }
    )
    df[[id_column, prediction_column]].to_csv(file_name, index=False)
    print("Output complete:", file_name, "rows:", len(df))


def plot_loss_accuracy_rmse(history):
    plt.figure(figsize=(20, 10))
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.title("model loss")
    plt.ylabel("loss")
    plt.xlabel("epoch")
    plt.legend(["train", "val"], loc="upper right")
    plt.show()

    if "rmse" in history.history:
        plt.figure(figsize=(20, 10))
        plt.plot(history.history["rmse"])
        plt.plot(history.history["val_rmse"])
        plt.title("Model rmse")
        plt.ylabel("rmse")
        plt.xlabel("epoch")
        plt.legend(["train", "val"], loc="upper right")
        plt.show()




In [8]:
print("train_df clean")
train_df = clean(train_df)
print("validation_df clean")
validation_df = clean(validation_df)

print("testKaggle clean")
testKaggle = clean(testKaggle)



train_df clean
 Old size: 225000
 New size after dropna: 224998
 New size after removing same pickup/dropoff point: 218571
 New size after removing 0 long lat: 218152
 New size after only NYC: 217891
 New size after removing outliers: 215215
 New size after removing 6=>passenger_count > 0 : 214409
 Skipping landmark point-noise removal (was overly aggressive).
Old size: 214409
New size: 214409
 New size: 214409
validation_df clean
 Old size: 25000
 New size after dropna: 25000
 New size after removing same pickup/dropoff point: 24270
 New size after removing 0 long lat: 24227
 New size after only NYC: 24202
 New size after removing outliers: 23910
 New size after removing 6=>passenger_count > 0 : 23837
 Skipping landmark point-noise removal (was overly aggressive).
Old size: 23837
New size: 23837
 New size: 23837
testKaggle clean
 Old size: 9914
 New size after dropna: 9914
 New size after removing same pickup/dropoff point: 9624
 New size after removing 0 long lat: 9596
 New size afte

In [9]:
train_df.describe()



,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000
mean,10.734820,-73.976440,40.751560,-73.974770,40.751835,1.687788
std,7.744761,0.039039,0.036600,0.039079,0.040068,1.304251
min,0.010000,-74.356125,40.509960,-74.341064,40.505650,1.000000
25%,6.000000,-73.992302,40.736980,-73.991486,40.736187,1.000000
50%,8.500000,-73.982140,40.753700,-73.980568,40.754154,1.000000
75%,12.500000,-73.968735,40.767670,-73.965614,40.768452,2.000000
max,50.000000,-72.856972,41.692429,-72.854942,41.714634,6.000000


In [10]:
validation_df.describe()



,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000
mean,10.683747,-73.976456,40.751640,-73.975227,40.751610,1.695683
std,7.617523,0.035310,0.027607,0.032432,0.030439,1.309320
min,2.500000,-74.295540,40.511086,-74.293663,40.517769,1.000000
25%,6.000000,-73.992378,40.736885,-73.991646,40.735512,1.000000
50%,8.500000,-73.982109,40.753910,-73.980850,40.754032,1.000000
75%,12.500000,-73.968987,40.767700,-73.965981,40.768326,2.000000
max,50.000000,-73.133446,41.366138,-73.137390,41.366138,6.000000


In [11]:
print("train_df add_time_features")
train_df = add_time_features(train_df)
print("validation_df add_time_features")
validation_df = add_time_features(validation_df)
print("testKaggle add_time_features")
testKaggle = add_time_features(testKaggle)



train_df add_time_features
validation_df add_time_features
testKaggle add_time_features


In [12]:
print("train_df add_coordinate_features")
train_df = add_coordinate_features(train_df)
print("validation_df add_coordinate_features")
validation_df = add_coordinate_features(validation_df)
print("testKaggle add_coordinate_features")
testKaggle = add_coordinate_features(testKaggle)



train_df add_coordinate_features
validation_df add_coordinate_features
testKaggle add_coordinate_features


In [13]:
print("train_df add_distances_features")
train_df = add_distances_features(train_df)
print("validation_df add_distances_features")
validation_df = add_distances_features(validation_df)
print("testKaggle add_distances_features")
testKaggle = add_distances_features(testKaggle)

# Score-relevant improvement (minimal): add airport + haversine distance features.
print("train_df add_airport_features")
train_df = add_airport_features(train_df)
print("validation_df add_airport_features")
validation_df = add_airport_features(validation_df)
print("testKaggle add_airport_features")
testKaggle = add_airport_features(testKaggle)

print("Done with Adding features")



train_df add_distances_features
validation_df add_distances_features
testKaggle add_distances_features
train_df add_airport_features


AttributeError: 'float' object has no attribute 'astype'

In [14]:
train_df.describe()



,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,year,month,day,hour,weekday,night,late_night,rush_hour,latdiff,londiff,manhattan,distance,haversine_km
count,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000,214409.000000
mean,10.734820,-73.976440,40.751560,-73.974770,40.751835,1.687788,2011.728394,6.274998,15.662580,13.507726,3.042326,0.244421,0.105905,0.200500,-0.000272,-0.001669,0.042028,0.032536,3.188318
std,7.744761,0.039039,0.036600,0.039079,0.040068,1.304251,3.475908,3.443581,8.673458,6.507473,1.946310,0.429744,0.307717,0.400375,0.030525,0.037560,0.046798,0.035873,3.373993
min,0.010000,-74.356125,40.509960,-74.341064,40.505650,1.000000,2009.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.617390,-1.082672,0.000004,0.000004,0.000424
25%,6.000000,-73.992302,40.736980,-73.991486,40.736187,1.000000,2010.000000,3.000000,8.000000,9.000000,1.000000,0.000000,0.000000,0.000000,-0.014343,-0.014069,0.016621,0.012991,1.272975
50%,8.500000,-73.982140,40.753700,-73.980568,40.754154,1.000000,2012.000000,6.000000,16.000000,14.000000,3.000000,0.000000,0.000000,0.000000,-0.000217,-0.001297,0.028126,0.021814,2.152691
75%,12.500000,-73.968735,40.767670,-73.965614,40.768452,2.000000,2013.000000,9.000000,23.000000,19.000000,5.000000,0.000000,0.000000,0.000000,0.013988,0.011421,0.050014,0.038119,3.849319
max,50.000000,-72.856972,41.692429,-72.854942,41.714634,6.000000,2015.000000,12.000000,31.000000,23.000000,6.000000,1.000000,1.000000,1.000000,0.868980,0.890816,1.626690,1.155449,110.833412


In [15]:
validation_df.describe()



,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,year,month,day,hour,weekday,night,late_night,rush_hour,latdiff,londiff,manhattan,distance
count,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000,23837.000000
mean,10.683747,-73.976456,40.751640,-73.975227,40.751610,1.695683,2011.734253,6.261442,15.709569,13.561355,3.032806,0.248731,0.106851,0.199522,0.000033,-0.001230,0.042150,0.032629
std,7.617523,0.035310,0.027607,0.032432,0.030439,1.309320,1.865620,3.430815,8.685909,6.535803,1.954474,0.432287,0.308930,0.399649,0.031280,0.038630,0.049299,0.037517
min,2.500000,-74.295540,40.511086,-74.293663,40.517769,1.000000,2009.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.638363,-0.841782,0.000004,0.000004
25%,6.000000,-73.992378,40.736885,-73.991646,40.735512,1.000000,2010.000000,3.000000,8.000000,9.000000,1.000000,0.000000,0.000000,0.000000,-0.014336,-0.014069,0.016747,0.013115
50%,8.500000,-73.982109,40.753910,-73.980850,40.754032,1.000000,2012.000000,6.000000,16.000000,14.000000,3.000000,0.000000,0.000000,0.000000,0.000008,-0.001007,0.028549,0.022115
75%,12.500000,-73.968987,40.767700,-73.965981,40.768326,2.000000,2013.000000,9.000000,23.000000,19.000000,5.000000,0.000000,0.000000,0.000000,0.014137,0.011734,0.050194,0.038286
max,50.000000,-73.133446,41.366138,-73.137390,41.366138,6.000000,2015.000000,12.000000,31.000000,23.000000,6.000000,1.000000,1.000000,1.000000,0.641449,0.871864,1.513313,1.082407


In [16]:
dropped_columns = ["passenger_count", "pickup_datetime"]

train_df = train_df.drop(dropped_columns + ["key"], axis=1)
validation_df = validation_df.drop(dropped_columns + ["key"], axis=1)
testKaggle_clean = testKaggle.drop(dropped_columns + ["key"], axis=1)

print("Done with dropped_columns")



Done with dropped_columns


In [17]:
for _df in (train_df, validation_df, testKaggle_clean):
    for c in _df.columns:
        if c != "fare_amount":
            if _df[c].dtype.kind in "fc":
                _df[c] = _df[c].fillna(0.0)
            else:
                _df[c] = _df[c].fillna(0)

train_labels = train_df["fare_amount"].values.astype("float32")
validation_labels = validation_df["fare_amount"].values.astype("float32")

if USE_LOG_TARGET:
    train_labels = np.log1p(train_labels)
    validation_labels = np.log1p(validation_labels)

train_df = train_df.drop(["fare_amount"], axis=1)
validation_df = validation_df.drop(["fare_amount"], axis=1)

print("Done with Labels")



Done with Labels


In [18]:
train_df.shape



(214409, 17)

In [19]:
validation_df.shape




(23837, 16)

In [20]:
def fit_clip_bounds(df, q_low=0.001, q_high=0.999):
    bounds = {}
    for c in df.columns:
        if df[c].dtype.kind in "fc":
            lo = float(df[c].quantile(q_low))
            hi = float(df[c].quantile(q_high))
            if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
                continue
            bounds[c] = (lo, hi)
    return bounds


def apply_clip_bounds(df, bounds):
    df = df.copy()
    for c, (lo, hi) in bounds.items():
        if c in df.columns:
            df[c] = df[c].clip(lower=lo, upper=hi)
    return df


clip_bounds = fit_clip_bounds(train_df, q_low=0.001, q_high=0.999)
train_df_clip = apply_clip_bounds(train_df, clip_bounds)
validation_df_clip = apply_clip_bounds(validation_df, clip_bounds)
testKaggle_clip = apply_clip_bounds(testKaggle_clean, clip_bounds)

scaler = preprocessing.MinMaxScaler()
train_df_scaled = scaler.fit_transform(train_df_clip)
validation_df_scaled = scaler.transform(validation_df_clip)
testKaggle_scaled = scaler.transform(testKaggle_clip)




ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- haversine_km


In [21]:
def rmse(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.sqrt(tf.reduce_mean(tf.square(y_pred - y_true), axis=-1))




In [22]:
model = Sequential()
model.add(
    Dense(
        256,
        activation="relu",
        input_dim=train_df_scaled.shape[1],
        activity_regularizer=regularizers.l1(0.01),
    )
)
model.add(BatchNormalization())
model.add(Dense(128, activation="relu"))
model.add(BatchNormalization())
model.add(Dense(64, activation="relu"))
model.add(BatchNormalization())
model.add(Dense(32, activation="relu"))
model.add(BatchNormalization())
model.add(Dense(8, activation="relu"))
model.add(BatchNormalization())
model.add(Dense(1))

adam = optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(loss="mean_squared_error", optimizer=adam, metrics=["mae", rmse])

print("Dataset size: %s" % DATASET_SIZE)
print("Epochs: %s" % EPOCHS)
print("Learning rate: %s" % LEARNING_RATE)
print("Batch size: %s" % BATCH_SIZE)
print("Input dimension: %s" % train_df_scaled.shape[1])
print("Features used: %s" % list(train_df.columns))
model.summary()

history = model.fit(
    x=train_df_scaled,
    y=train_labels,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1,
    validation_data=(validation_df_scaled, validation_labels),
    shuffle=True,
)



Dataset size: 250000
Epochs: 30
Learning rate: 0.001
Batch size: 256
Input dimension: 17
Features used: ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'year', 'month', 'day', 'hour', 'weekday', 'night', 'late_night', 'rush_hour', 'latdiff', 'londiff', 'manhattan', 'distance', 'haversine_km']
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 256)               4608      
                                                                 
 batch_normalization (Batch  (None, 256)               1024      
 Normalization)                                                  
                                                                 
 dense_1 (Dense)             (None, 128)               32896     
                                                                 
 batch_normalization_1 (Bat  (None, 128)               512 

I0000 00:00:1768689676.244907      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:06:00.0, compute capability: 8.6


NameError: name 'validation_df_scaled' is not defined

In [23]:
print(
    "Model visualization skipped (keras.utils.vis_utils not available / not needed for submission)."
)



Model visualization skipped (keras.utils.vis_utils not available / not needed for submission).


In [24]:
plot_loss_accuracy_rmse(history)



NameError: name 'history' is not defined

In [25]:
predictionKaggle = model.predict(testKaggle_scaled, batch_size=128, verbose=1)



NameError: name 'testKaggle_scaled' is not defined

In [26]:
predictionKaggle = predictionKaggle.reshape(-1).astype("float32")
if USE_LOG_TARGET:
    predictionKaggle = np.expm1(predictionKaggle)

predictionKaggle = np.maximum(predictionKaggle, 0.0).reshape(-1, 1)
output_submission(testKaggle, predictionKaggle, "key", "fare_amount", SUBMISSION_PATH)



NameError: name 'predictionKaggle' is not defined

In [27]:
print("Wrote submission to:", SUBMISSION_PATH)
print("Submission preview:")
print(pd.read_csv(SUBMISSION_PATH).head())

Wrote submission to: /kaggle/working/submissiontry_water.csv
Submission preview:


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/submissiontry_water.csv'